# 제353회~제414회 회의록 요구자료 추출

엑셀 형식(.xlsx)으로 저장된 회의록에서 자료 제출 요구 내용을 추출합니다.

## 입력 구조
```
data_parsed_by_meeting/
├── 제353회/
│   ├── 위_제1차__2017__08__21__.xlsx
│   ├── 소_안전및선거법심사소위원회_제1차__2017__08__23__.xlsx
│   └── ...
├── 제354회/
└── ...
```

## 출력 형식
- 회의회차, 요구자명, 요구자정당, 대상, 실제요구자료, 카테고리

In [1]:
import os
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

In [3]:
# OpenAI 클라이언트 초기화 (API 키는 환경변수에서 자동으로 읽어옴)

from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)

API_KEY = os.getenv("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY 로드 실패")
    
client = OpenAI(
    api_key=API_KEY
)  # OPENAI_API_KEY 환경변수에 세팅되어 있어야 함
# 데이터 디렉토리 설정
DATA_DIR = "data_parsed_by_meeting"

# GPT 모델 설정
MODEL = "gpt-4-turbo"  # 또는 "gpt-4o-mini"
SLEEP_SEC = 0.25  # API 호출 간 대기 시간

## 핵심 함수 정의

In [4]:
def is_candidate_request(text: str) -> bool:
    """
    발언 내용에 자료 요구 키워드가 포함되어 있는지 빠르게 필터링
    """
    keywords = [
        "제출", "요구", "자료", "제출해", "요청", "보고", "명단", 
        "내역", "현황", "통계", "자료 주시", "자료를"
    ]
    return any(kw in text for kw in keywords)

In [5]:
def extract_requests_from_speech(text: str, model: str = "gpt-4-turbo") -> list:
    """
    GPT 모델을 이용해 발언 내용에서 실제 자료 요구만 추출
    """
    system_prompt = """당신은 국회 회의록 분석 전문가입니다.
발언 내용을 분석하여 **실제로 자료 제출을 요구한 내용**만 추출하세요.

**추출 기준:**
- 명확하게 "제출해 주시기 바랍니다", "자료를 요청합니다", "보고해 주십시오" 등의 표현이 있는 경우
- 구체적인 자료명이나 내용이 명시된 경우

**제외 기준:**
- 일반적인 질문이나 의견 제시
- 단순히 "자료"라는 단어가 언급된 경우
- 과거에 제출된 자료를 언급하는 경우

**출력 형식:**
각 요구 건마다 JSON 객체로 반환:
{
  "target": "자료 제출 대상 기관/부처",
  "requested_data": "요구하는 자료의 구체적인 내용",
  "category": "재난·안전|예산·재정|통계·현황|정책·제도|인사·조직|감사·점검|기타 중 하나"
}

여러 건이 있으면 JSON 배열로 반환하세요."""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"다음 발언에서 자료 요구 내용을 추출하세요:\n\n{text}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        result_text = response.choices[0].message.content.strip()
        parsed = json.loads(result_text)
        
        # 결과가 단일 객체인 경우 리스트로 변환
        if isinstance(parsed, dict):
            if "requests" in parsed:
                return parsed["requests"]
            else:
                return [parsed] if parsed.get("requested_data") else []
        elif isinstance(parsed, list):
            return parsed
        else:
            return []
            
    except Exception as e:
        print(f"GPT 추출 오류: {e}")
        return []

In [6]:
def build_requester_party_map(df: pd.DataFrame) -> dict:
    """
    DataFrame에서 발언자명과 소속(정당)을 매핑
    소속이 '더불어민주당', '국민의힘' 등 정당명이 포함된 경우만 추출
    """
    party_map = {}
    
    for _, row in df.iterrows():
        speaker = str(row.get("발언자", "")).strip()
        affiliation = str(row.get("소속", "")).strip()
        
        if not speaker or not affiliation:
            continue
        
        # 정당 키워드 체크
        party_keywords = ["더불어민주", "국민의힘", "국민의당", "정의당", "바른미래당", 
                         "자유한국당", "새누리당", "민주당", "한국당", "열린민주당"]
        
        is_party = any(kw in affiliation for kw in party_keywords)
        
        if is_party:
            party_map[speaker] = affiliation
        else:
            # 정당이 아닌 경우 공백 처리
            party_map[speaker] = ""
    
    return party_map


def get_requester_party(party_map: dict, requester_name: str) -> str:
    """
    요구자명에 해당하는 정당 반환
    """
    return party_map.get(str(requester_name).strip(), "")

In [7]:
def extract_meeting_round_from_folder(folder_name: str) -> str:
    """
    폴더명에서 회차 추출
    '제353회' -> '353회'
    """
    match = re.search(r'제(\d+)회', folder_name)
    if match:
        return f"{match.group(1)}회"
    return folder_name


def extract_meeting_round_from_text(meeting_number: str) -> str:
    """
    회의 번호 텍스트에서 회차 추출
    '제20대국회 제353회 (임시회) 제1차 행정안전위원회' -> '353회'
    """
    match = re.search(r'제(\d+)회', meeting_number)
    if match:
        return f"{match.group(1)}회"
    return ""

## 메인 추출 파이프라인

In [8]:
def run_request_extraction_pipeline(
    data_dir: str,
    model: str = "gpt-4-turbo",
    sleep_sec: float = 0.25
) -> pd.DataFrame:
    """
    data_dir 하위의 '제353회' ~ '제414회' 폴더들을 순회하며,
    각 엑셀 파일에서 발언자별 발언을 합쳐 '실제 자료 제출 요구'만 추출.
    
    출력 컬럼:
      회의회차, 요구자명, 요구자정당, 대상, 실제요구자료, 카테고리
    """
    all_results = []
    candidate_count = 0
    
    # 회차 폴더 목록 가져오기 (제353회 ~ 제414회)
    folders = [f for f in os.listdir(data_dir) if f.startswith("제") and f.endswith("회")]
    folders.sort(key=lambda x: int(re.search(r'\d+', x).group()))
    
    print(f"총 {len(folders)}개 회차 폴더 발견")
    
    for folder in folders:
        folder_path = os.path.join(data_dir, folder)
        
        if not os.path.isdir(folder_path):
            continue
        
        meeting_round = extract_meeting_round_from_folder(folder)
        print(f"\n▶ 처리 중: {meeting_round} (폴더: {folder})")
        
        # 폴더 내 모든 엑셀 파일 가져오기
        excel_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]
        
        if not excel_files:
            print(f"  ⚠ 엑셀 파일 없음")
            continue
        
        print(f"  {len(excel_files)}개 회의 파일 발견")
        
        for excel_file in excel_files:
            file_path = os.path.join(folder_path, excel_file)
            
            try:
                # 엑셀 파일 읽기
                df = pd.read_excel(file_path).fillna("")
                
                if len(df) == 0:
                    continue
                
                # 정당 매핑 생성
                party_map = build_requester_party_map(df)
                
                # 발언자별로 발언 합치기
                grouped = (
                    df.groupby("발언자")["발언 내용"]
                    .apply(lambda x: " ".join([str(t) for t in x if str(t).strip()]))
                    .reset_index()
                    .rename(columns={"발언 내용": "speeches"})
                )
                
                # 각 발언자별로 처리
                for _, row in tqdm(grouped.iterrows(), total=len(grouped), 
                                  desc=f"  {excel_file[:30]}"):
                    requester = str(row["발언자"]).strip()
                    speeches = str(row["speeches"]).strip()
                    
                    if not requester or not speeches:
                        continue
                    
                    # 1단계: 후보 필터
                    if not is_candidate_request(speeches):
                        continue
                    
                    candidate_count += 1
                    
                    # 2단계: GPT 정밀 추출
                    requests = extract_requests_from_speech(speeches, model=model)
                    
                    # 요구자 정당 매핑
                    requester_party = get_requester_party(party_map, requester)
                    
                    for req in requests:
                        all_results.append({
                            "회의회차": meeting_round,
                            "요구자명": requester,
                            "요구자정당": requester_party,
                            "대상": req.get("target", ""),
                            "실제요구자료": req.get("requested_data", ""),
                            "카테고리": req.get("category", "")
                        })
                    
                    time.sleep(sleep_sec)
                    
            except Exception as e:
                print(f"  ⚠ 파일 처리 오류 ({excel_file}): {e}")
                continue
    
    print(f"\n\n=== 처리 완료 ===")
    print(f"후보 발언 수: {candidate_count}")
    print(f"추출 결과 수: {len(all_results)}")
    
    return pd.DataFrame(all_results)

## 실행

In [9]:
# 추출 실행
result_df = run_request_extraction_pipeline(
    data_dir=DATA_DIR,
    model=MODEL,
    sleep_sec=SLEEP_SEC
)

총 48개 회차 폴더 발견

▶ 처리 중: 353회 (폴더: 제353회)
  5개 회의 파일 발견


  위 제2차 (2017. 08. 23.).xlsx: 100%|█████████████████████████████████████████████████| 19/19 [02:51<00:00,  9.05s/it]



▶ 처리 중: 354회 (폴더: 제354회)
  14개 회의 파일 발견


  위 제6차 (2017. 11. 30.).xlsx: 100%|█████████████████████████████████████████████████| 19/19 [00:39<00:00,  2.10s/it]



▶ 처리 중: 355회 (폴더: 제355회)
  5개 회의 파일 발견


...

GPT 추출 오류: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  위 제1차 (2024. 05. 02.).xlsx:  33%|█████████████████                                  | 2/6 [00:04<00:08,  2.15s/it]

GPT 추출 오류: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  위 제1차 (2024. 05. 02.).xlsx: 100%|███████████████████████████████████████████████████| 6/6 [00:06<00:00,  1.09s/it]



=== 처리 완료 ===
후보 발언 수: 3733
추출 결과 수: 480


In [10]:
# 결과 미리보기
print(f"총 추출 건수: {len(result_df)}")
result_df.head(20)

총 추출 건수: 480


,회의회차,요구자명,요구자정당,대상,실제요구자료,카테고리
0,353회,박성중,,"인사혁신처, 경찰청, 소방청",조직의 효율성 관점에서 전반적인 검토 및 개선 방안,인사·조직
1,353회,진선미,,위원장,법안에 대한 체계 및 자구 정리,정책·제도
2,353회,표창원,,"기획재정부, 인사혁신처",2년 격차 추가 해소에 대한 논의,정책·제도
3,353회,김부년,,행정안전부,보전금 예산의 집행근거를 마련하고 철저한 집행관리를 할 것,예산·재정
4,353회,김엽,,기재부,적정예산 책정을 위한 협의 및 조치,예산·재정
5,353회,김영호,,국민안전처,지진 발생 시 국민안전처 홈페이지 다운 원인과 관련하여 서버 용량을 많이 차지하는 ...,재난·안전
6,353회,김혜순,,해당 부처,재정소요 추계자료,예산·재정
7,353회,김홍필,,행안부,내년도 소요예산에 4명 채용하기로 합의한 자료,인사·조직
8,353회,류희인,,행정안전부,지진가속도계측기 설치현황,재난·안전
9,353회,박남춘,,행정안전부,세종청사 이전과 관련한 미래부 행안부 이전과 관련한 자료,정책·제도


In [11]:
# 결과 저장
output_path = "요구자료_목록_353_414회_통합.csv"
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {output_path}")


저장 완료: 요구자료_목록_353_414회_통합.csv


## 결과 분석 (선택사항)

In [12]:
# 회차별 요구자료 건수
print("\n=== 회차별 요구자료 건수 ===")
print(result_df["회의회차"].value_counts().sort_index())


=== 회차별 요구자료 건수 ===
회의회차
353회     65
354회    177
355회     52
356회     66
358회      8
360회     18
362회     49
363회     36
364회      8
410회      1
Name: count, dtype: int64


In [13]:
# 카테고리별 분포
print("\n=== 카테고리별 분포 ===")
print(result_df["카테고리"].value_counts())


=== 카테고리별 분포 ===
카테고리
정책·제도    176
재난·안전    102
감사·점검     76
예산·재정     59
인사·조직     37
통계·현황     26
기타         4
Name: count, dtype: int64


In [14]:
# 요구자별 건수 (상위 20명)
print("\n=== 요구자별 건수 (상위 20명) ===")
print(result_df["요구자명"].value_counts().head(20))


=== 요구자별 건수 (상위 20명) ===
요구자명
권은희    25
김부년    23
윤재옥    22
박성중    17
진선미    17
이재정    16
유민봉    16
김영진    15
이용호    14
표창원    14
소병훈    13
홍철호    13
柳在仲    13
심보균    12
이명수    12
김부겸    12
황영철    11
박남춘    11
김영호    10
박순자    10
Name: count, dtype: int64


In [15]:
# 정당별 건수
print("\n=== 정당별 건수 ===")
party_counts = result_df[result_df["요구자정당"] != ""]["요구자정당"].value_counts()
print(party_counts)


=== 정당별 건수 ===
Series([], Name: count, dtype: int64)
